In [1]:
import os, sys
import numpy as np
import pandas as pd
import clip
from glob import glob
import torch
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(root)
from utils import utils,models
from PIL import Image

In [2]:
def resize_with_transparent_padding(
    img: Image.Image,
    target_w: int = 460,
    target_h: int = 260
):

    # Convert to RGBA for transparency support
    img = img.convert("RGBA")

    original_w, original_h = img.size

    # Preserve aspect ratio
    ratio = min(target_w / original_w, target_h / original_h)

    new_w = round(original_w * ratio)
    new_h = round(original_h * ratio)

    resized_img = img.resize(
        (new_w, new_h),
        Image.Resampling.LANCZOS
    )

    # Fully transparent canvas
    canvas = Image.new(
        "RGBA",
        (target_w, target_h),
        (0, 0, 0, 0)  # transparent
    )

    # Center placement
    paste_x = (target_w - new_w) // 2
    paste_y = (target_h - new_h) // 2

    canvas.paste(resized_img, (paste_x, paste_y))

    return canvas

In [3]:
device = utils.get_device()
print(f"Using device: {device}")
batch_size = 64

Using device: mps


In [4]:
model_name = "ViT-L/14"
model, preprocess = utils.load_model(model_name, device)

In [6]:
dataset_dir = '../dataset'
images_dir = f'{dataset_dir}/yfcc100m_50k'
image_paths = glob(os.path.join(images_dir, '*'))
test_article_titles = pd.read_csv(f'{dataset_dir}/newsimages_test_and_evaluation_26_v1.0/news_articles_test.csv',encoding='latin1')
test_article_titles.set_index('article_id', inplace=True)
test_article_titles['article_title'] = test_article_titles['article_title'].apply(utils.normalize_text)
test_article_titles = test_article_titles[['article_title']]

In [7]:
test_article_titles.head()

,article_title
article_id,
8501,The unveiling of the Iltis Monument in Shanghai.
8504,BENIGN DISEASE EISENHOWER'S CONDITION VERY SAT...
8509,Johnson accepts surprising offer from governme...
8511,Landslide French elections De Gaulle's great t...
8516,British-Dutch cooperation MARINE DEPLOYED ON O...


In [ ]:
image_embeddings = np.load("../artifacts/ViT-L-14_embeddings_pool_images.npy")
reranker_image_embeddings = np.load("../artifacts/ViT-L-14@336px_embeddings_pool_images.npy")

In [9]:
group_name = 'FAST-DS'
output_dir = f'../output/{group_name}_Submission'

# RUN 1 - Baseline OpenCLIP: VisionAnchor

In [ ]:
approach_name = 'VisionAnchor'
run_dir = f"{output_dir}/{group_name}_{approach_name}"
os.makedirs(run_dir, exist_ok=True)

baselineOpenClip = model

for index,row in test_article_titles.iterrows():

    query = row['article_title']
    
    candidates = utils.retrieve_candidate(query, baselineOpenClip, device, image_embeddings, image_paths)
    
    image = Image.open(candidates[0]['image_path'])
    
    resized = resize_with_transparent_padding(image)
    
    output_image_path = f"{run_dir}/{index}_{group_name}_{approach_name}"
    
    resized.save(f"{output_image_path}.png")

# RUN 2 - OpenCLIP + Reranking Refinement: VisionAnchor-R

In [ ]:
approach_name = 'VisionAnchor-R'
reranker_model = 'ViT-L/14@336px'

run_dir = f"{output_dir}/{group_name}_{approach_name}"
os.makedirs(run_dir, exist_ok=True)

base_retriever = models.CLIPRetrieval(
    model=model,
    image_embeddings=image_embeddings,
    device=device
)

rerank_retriever = models.RerankRetrieval(
    retriever=base_retriever,
    reranker=reranker_model,
    reranker_image_embeddings=reranker_image_embeddings,
    device=device,
    k=50
)

for index, row in test_article_titles.iterrows():

    query = row['article_title']

    candidate_ids = rerank_retriever.retrieve(
        query=query,
        k_final=1
    )

    image_path = image_paths[candidate_ids[0]]

    image = Image.open(image_path)

    resized = resize_with_transparent_padding(image)

    output_image_path = (
        f"{run_dir}/{index}_{group_name}_{approach_name}.png"
    )

    resized.save(output_image_path)

# RUN 3 - OpenCLIP + Phi-3 Query Expansion: QueryForge

In [ ]:
approach_name = 'QueryForge'

run_dir = f"{output_dir}/{group_name}_{approach_name}"

os.makedirs(run_dir, exist_ok=True)

generator = models.OllamaPipeline(model="phi3")

qe_retriever = models.QERetrieval(

    model=model,

    image_embeddings=image_embeddings,

    generator=generator,

    device=device,

    cache_path="./qe_cache.json"

)

for index, row in test_article_titles.iterrows():

    query = row['article_title']

    candidate_ids = qe_retriever.retrieve(query, k=1)

    image_path = image_paths[candidate_ids[0]]

    image = Image.open(image_path)

    resized = resize_with_transparent_padding(image)

    output_image_path = (

        f"{run_dir}/{index}_{group_name}_{approach_name}.png"

    )

    resized.save(output_image_path)

# RUN 4 - OpenCLIP + Phi-3 QE + OpenCLIP Reranking):  QueryForge-RX

In [ ]:
approach_name = 'QueryForge-RX'

run_dir = f"{output_dir}/{group_name}_{approach_name}"
os.makedirs(run_dir, exist_ok=True)

generator = models.OllamaPipeline(model="phi3")

qe_base_retriever = models.QERetrieval(
    model=model,
    image_embeddings=image_embeddings,
    generator=generator,
    device=device,
    cache_path="./qe_cache.json"
)

qe_rerank_retriever = models.RerankRetrieval(
    retriever=qe_base_retriever,
    reranker=reranker_model,
    reranker_image_embeddings=reranker_image_embeddings,
    device=device,
    k=50
)

for index, row in test_article_titles.iterrows():

    query = row['article_title']

    candidate_ids = qe_rerank_retriever.retrieve(
        query=query,
        k_final=1
    )

    image_path = image_paths[candidate_ids[0]]

    image = Image.open(image_path)

    resized = resize_with_transparent_padding(image)

    output_image_path = (
        f"{run_dir}/{index}_{group_name}_{approach_name}.png"
    )

    resized.save(output_image_path)